In [23]:
import pandas as pd
import numpy as np
import json
import re

df = pd.read_csv("super_dirty_students.csv")
print(df.head(10))

# column names
print(df.columns)

# number of columns
print(len(df.columns))

# missing values count
print(df.isnull().sum())

# data types
df.dtypes

   student_id                name       age  gender   score  \
0           1      Claudia Short         20     NaN     NaN   
1           2                 NaN        20  Female  ninety   
2           3      Kathryn Moyer         20     NaN  ninety   
3           4        Ruben Wilson    twenty   fmale      81   
4           5       Robert Pruitt        20  Female     NaN   
5           6      David Martinez        20    Male  ninety   
6           7                 NaN  18 years  FEMALE      71   
7           8     Nicholas Dennis        21  FEMALE  ninety   
8           9                 NaN    twenty   fmale      77   
9          10  Elizabeth Villegas        22  femlae     NaN   

                    phone               city                     email  \
0     +1-619-379-4152x102          Katieland          someonegmail.com   
1                     NaN          Dawnburgh         psmith@chen.com     
2                     NaN   Lake Stevenmouth                       NaN   
3         

student_id      int64
name              str
age               str
gender            str
score             str
phone             str
city              str
email             str
date_of_join      str
course            str
attendance        str
status            str
gpa               str
remarks           str
money_spent       str
event_time        str
address_raw       str
profile_json      str
dtype: object

In [ ]:
# Step 2: Oddiy string tozalash
# Har bir string columnning bo‘sh joylarini olib tashlash (strip).
# Bo‘sh stringlarni NaN yoki Null bilan almashtirish.
# string_cols = df.select_dtypes(include=['object', 'string']).columns

string_cols = df.select_dtypes(include=['object', 'string']).columns

for col in string_cols:
    # 2. Avval ma'lumotni string ko'rinishiga keltiramiz
    df[col] = df[col].astype(str)
    
    # 3. Bo'sh joylarni olib tashlaymiz
    df[col] = df[col].str.strip()
    
    # 4. Keraksiz so'zlarni haqiqiy bo'sh qiymat (NaN) ga aylantiramiz
    # Bu yerda case=False qilsak, 'none', 'NONE' kabi variantlarni ham ushlaydi
    df[col] = df[col].replace(['nan', 'None', 'NULL', '', 'None', 'nan'], np.nan)

# Tekshirish uchun
print("String ustunlar tozalandi!")

String ustunlar tozalandi!


In [25]:
import ast
import pandas as pd
import numpy as np
import re

word_to_num = {
    'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 
    'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
    'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50, 'sixty': 60,
    'seventy': 70, 'eighty': 80, 'ninety': 90, 'point': '.'
}

def robust_numeric(val):
    if pd.isna(val) or val == 'nan': return np.nan
    val = str(val).lower().strip().replace(',', '.')
    
    # 1. Regex orqali raqamni qidirish
    match = re.search(r'[-+]?\d*\.?\d+', val)
    if match: return float(match.group())
    
    # 2. So'z bilan yozilgan bo'lsa
    words = re.findall(r'[a-z]+', val)
    num_str = ""
    for w in words:
        if w in word_to_num: num_str += str(word_to_num[w])
    try:
        return float(num_str) if num_str else np.nan
    except: return np.nan

# --- Raqamlarni tozalash ---
df['age'] = df['age'].apply(robust_numeric)
df['score'] = df['score'].apply(robust_numeric)
df['gpa'] = df['gpa'].apply(robust_numeric)
df['attendance'] = df['attendance'].apply(robust_numeric)
df['money_spent'] = df['money_spent'].astype(str).str.replace(',', '.').apply(lambda x: re.sub(r'[^0-9.]', '', x)).replace('', np.nan).astype(float)

# --- Sanalarni tozalash (Warninglarsiz) ---
def clean_date_column(series):
    # Agar qiymat Unix timestamp bo'lsa (uzun raqam)
    def parse_single_date(x):
        if pd.isna(x): return pd.NaT
        try:
            # Agar bu 10 xonali raqam bo'lsa (timestamp)
            if str(x).isdigit() and len(str(x)) >= 10:
                return pd.to_datetime(int(x), unit='s')
        except:
            pass
        # Boshqa hollarda formatni avtomatik aniqlash (mixed formatlar uchun)
        return pd.to_datetime(x, errors='coerce', dayfirst=False)

    return series.apply(parse_single_date)

df['date_of_join'] = clean_date_column(df['date_of_join'])
df['event_time'] = clean_date_column(df['event_time'])

print("Step 3: RStep 3 bajarildi")

C:\Users\hp\AppData\Local\Temp\ipykernel_4932\1195637293.py:49: UserWarning: Parsing dates in %d/%m/%Y %I:%M %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(x, errors='coerce', dayfirst=False)
C:\Users\hp\AppData\Local\Temp\ipykernel_4932\1195637293.py:49: UserWarning: Parsing dates in %d/%m/%Y %I:%M %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(x, errors='coerce', dayfirst=False)


Step 3: RStep 3 bajarildi


In [26]:
def clean_email(email):
    if pd.isna(email) or '@' not in str(email): return np.nan
    email = str(email).lower().strip()
    return email if email.count('@') == 1 else np.nan

df['email'] = df['email'].apply(clean_email)

def clean_phone(phone):
    if pd.isna(phone): return np.nan
    nums = re.sub(r'\D', '', str(phone)) # Faqat raqamlarni qoldirish
    if len(nums) >= 9:
        return "+998" + nums[-9:] # Oxirgi 9 ta raqamni olib standartlash
    return np.nan

df['phone'] = df['phone'].apply(clean_phone)

In [27]:
def parse_profile(json_str):
    if pd.isna(json_str) or json_str == 'INVALID_JSON_DATA': return {}
    try:
        # Ba'zan JSON bir tirnoqli bo'ladi, ast.literal_eval shuni o'qiydi
        return ast.literal_eval(json_str)
    except:
        return {}

profile_data = pd.json_normalize(df['profile_json'].apply(parse_profile))
# Asosiy df bilan birlashtirish
df = pd.concat([df.reset_index(drop=True), profile_data.reset_index(drop=True)], axis=1)

In [28]:
def parse_address(addr):
    if pd.isna(addr) or "BROKEN" in str(addr): 
        return pd.Series([None, None, None])
    
    city = 'Tashkent' if 'Tashkent' in str(addr) else 'Other'
    zip_code = re.findall(r'\d{6}', str(addr))
    dist_match = re.search(r'([^,]+)\sdistrict', str(addr))
    
    return pd.Series([
        city, 
        dist_match.group(1).strip() if dist_match else None, 
        zip_code[0] if zip_code else None
    ])

df[['addr_city', 'addr_district', 'addr_postal']] = df['address_raw'].apply(parse_address)

In [29]:
# Student_id bo'yicha duplikatlarni olib tashlash
df = df.drop_duplicates(subset=['student_id'], keep='first')
# Muhim bo'shliqlarni tozalash (ixtiyoriy)
# df = df.dropna(subset=['student_id'])

In [30]:
# Gender normalizatsiya
df['gender'] = df['gender'].astype(str).str.capitalize()
df.loc[df['gender'].str.contains('Fem|Fm', na=False), 'gender'] = 'Female'
df.loc[df['gender'].str.contains('Male|M', na=False) & ~df['gender'].str.contains('Fe', na=False), 'gender'] = 'Male'
df.loc[~df['gender'].isin(['Male', 'Female']), 'gender'] = 'Unknown'

# Course normalizatsiya
def map_course(val):
    val = str(val).lower()
    if 'data' in val or 'ds' in val: return 'Data Science'
    if 'python' in val: return 'Python'
    return 'Other'

df['course'] = df['course'].apply(map_course)

In [32]:
import pandas as pd

# 1. date_of_join ustunini to'g'rilash (Unix timestamp bo'lgani uchun unit='s' qo'shamiz)
df['date_of_join'] = pd.to_datetime(df['date_of_join'], unit='s', errors='coerce')

# 2. event_time ustunini to'g'rilash
# Agar bu ham son bo'lsa unit='s' qoldiring, agar oddiy tekst bo'lsa unitni olib tashlang
df['event_time'] = pd.to_datetime(df['event_time'], unit='s', errors='coerce')

# 3. Formatni o'zgartirish (String ko'rinishiga keltirish)
df['date_of_join'] = df['date_of_join'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['event_time'] = df['event_time'].dt.strftime('%Y-%m-%d %H:%M:%S')

# 4. Saqlash
df.to_csv('cleaned_students_3.csv', index=False)
print("Fayl muvaffaqiyatli saqlandi!")


Fayl muvaffaqiyatli saqlandi!


In [33]:
import pandas as pd

# Faylni o'qish
df = pd.read_csv("super_dirty_students.csv")

# 1. Umumiy dublikat qatorlar sonini tekshirish
duplicate_count = df.duplicated().sum()
print(f"Jami dublikat qatorlar soni: {duplicate_count}")

# 2. Agar dublikat bo'lsa, ularni ko'rib olish
if duplicate_count > 0:
    duplicates = df[df.duplicated(keep=False)] # keep=False barcha takrorlanganlarini ko'rsatadi
    print("\nDublikat qatorlar namunalari:")
    print(duplicates.head())
else:
    print("Dublikatlar topilmadi!")

Jami dublikat qatorlar soni: 0
Dublikatlar topilmadi!
